# Retention Curves

## Overview
**Retention curves** show what fraction of a starting population is still active at each point in time. They complement cohort tables by revealing the shape of engagement loss -- fast early drop-off vs gradual long-term churn.

**Retention definitions:**

| Type | Definition |
|---|---|
| **Classic (N-day / N-month)** | % of users active on exactly day/month N after acquisition |
| **Rolling / cumulative** | % of users active at any point up through day/month N |
| **Unbounded** | % of users who were ever active after day N (used less often) |

**Churn vs retention:**
- Retention rate (month N) = active_in_month_N / cohort_size
- Churn rate (month N) = 1 - retention_rate(month N)
- Monthly churn identifies when customers typically disengage

**Reading a retention curve:**
- Steep initial drop then flat line = early friction, then loyal core
- Gradual steady decline = no clear drop-off, consistent churn
- Rising curve = impossible (users cannot un-churn in classic retention)

**PostgreSQL notes:** `GENERATE_SERIES` creates a date spine for gap-free monthly periods. In SQLite, use a CTE with recursive UNION ALL (shown in `05_subqueries_ctes/recursive_ctes`).

---

In [ ]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT, segment TEXT, province TEXT, signup_date TEXT, active INTEGER DEFAULT 1);CREATE TABLE accounts (account_id INTEGER PRIMARY KEY, customer_id INTEGER, account_type TEXT, opened_date TEXT, status TEXT, credit_limit REAL);CREATE TABLE transactions (txn_id INTEGER PRIMARY KEY, account_id INTEGER, customer_id INTEGER, txn_date TEXT, txn_type TEXT, amount REAL, category TEXT);CREATE TABLE product_events (event_id INTEGER PRIMARY KEY, customer_id INTEGER, event_date TEXT, event_type TEXT, product TEXT, channel TEXT);INSERT INTO customers VALUES (1,'Aroha','Ngata','Retail','NB','2023-01-05',1),(2,'Liam','Chen','Wealth','NS','2023-01-12',1),(3,'Fatima','Al-Rashid','SME','ON','2023-01-20',1),(4,'James','MacLeod','Retail','NB','2023-02-03',1),(5,'Sofia','Petrov','Retail','BC','2023-02-14',1),(6,'Noah','Williams','SME','AB','2023-02-28',0),(7,'Mei','Zhang','Wealth','ON','2023-03-10',1),(8,'Ethan','Tremblay','Retail','QC','2023-04-05',1),(9,'Priya','Nair','Wealth','BC','2023-04-18',1),(10,'Marcus','Okafor','SME','ON','2023-04-25',1),(11,'Diana','Flores','Retail','NB','2023-05-02',1),(12,'Ravi','Patel','Retail','ON','2023-05-15',1),(13,'Yuki','Tanaka','Wealth','BC','2023-06-01',1),(14,'Omar','Hassan','SME','QC','2023-06-20',1),(15,'Chloe','Bouchard','Retail','QC','2023-07-08',0);INSERT INTO accounts VALUES (101,1,'Chequing','2023-01-05','Active',NULL),(102,1,'Savings','2023-01-05','Active',NULL),(103,2,'Investment','2023-01-12','Active',50000),(104,3,'Chequing','2023-01-20','Active',NULL),(105,3,'Loan','2023-01-20','Active',25000),(106,4,'Chequing','2023-02-03','Active',NULL),(107,5,'Chequing','2023-02-14','Active',NULL),(108,5,'Savings','2023-02-14','Active',NULL),(109,6,'Chequing','2023-02-28','Closed',NULL),(110,7,'Investment','2023-03-10','Active',75000),(111,8,'Chequing','2023-04-05','Active',NULL),(112,9,'Investment','2023-04-18','Active',100000),(113,10,'Chequing','2023-04-25','Active',NULL),(114,10,'Loan','2023-04-25','Active',15000),(115,11,'Chequing','2023-05-02','Active',NULL),(116,12,'Chequing','2023-05-15','Active',NULL),(117,12,'Savings','2023-05-15','Active',NULL),(118,13,'Investment','2023-06-01','Active',60000),(119,14,'Chequing','2023-06-20','Active',NULL),(120,15,'Chequing','2023-07-08','Closed',NULL);INSERT INTO transactions VALUES (1001,101,1,'2023-01-10','Deposit',2500.00,'Payroll'),(1002,101,1,'2023-01-22','Withdrawal',-800.00,'Rent'),(1003,102,1,'2023-02-05','Deposit',500.00,'Transfer'),(1004,103,2,'2023-02-10','Deposit',10000.00,'Investment'),(1005,104,3,'2023-02-15','Deposit',3200.00,'Payroll'),(1006,104,3,'2023-03-01','Withdrawal',-1200.00,'Rent'),(1007,106,4,'2023-03-10','Deposit',2800.00,'Payroll'),(1008,107,5,'2023-03-15','Deposit',2200.00,'Payroll'),(1009,107,5,'2023-03-28','Fee',-25.00,'Monthly Fee'),(1010,101,1,'2023-03-10','Deposit',2500.00,'Payroll'),(1011,103,2,'2023-04-15','Deposit',15000.00,'Investment'),(1012,110,7,'2023-04-20','Deposit',20000.00,'Investment'),(1013,111,8,'2023-05-01','Deposit',2100.00,'Payroll'),(1014,112,9,'2023-05-10','Deposit',25000.00,'Investment'),(1015,113,10,'2023-05-20','Deposit',3500.00,'Payroll'),(1016,115,11,'2023-06-01','Deposit',2000.00,'Payroll'),(1017,116,12,'2023-06-15','Deposit',2700.00,'Payroll'),(1018,118,13,'2023-07-01','Deposit',18000.00,'Investment'),(1019,101,1,'2023-07-10','Deposit',2500.00,'Payroll'),(1020,103,2,'2023-07-20','Deposit',12000.00,'Investment'),(1021,104,3,'2023-07-15','Deposit',3200.00,'Payroll'),(1022,107,5,'2023-08-01','Deposit',2200.00,'Payroll'),(1023,111,8,'2023-08-05','Deposit',2100.00,'Payroll'),(1024,113,10,'2023-08-15','Withdrawal',-500.00,'Transfer'),(1025,116,12,'2023-08-20','Deposit',2700.00,'Payroll'),(1026,101,1,'2023-10-10','Deposit',2500.00,'Payroll'),(1027,116,12,'2023-10-20','Deposit',2700.00,'Payroll');INSERT INTO product_events VALUES (1,1,'2023-01-10','page_view','Savings Account','web'),(2,1,'2023-01-10','product_view','Savings Account','web'),(3,1,'2023-01-10','application_start','Savings Account','web'),(4,1,'2023-01-11','application_submit','Savings Account','web'),(5,1,'2023-01-15','account_opened','Savings Account','web'),(6,2,'2023-01-12','page_view','Investment Account','mobile'),(7,2,'2023-01-12','product_view','Investment Account','mobile'),(8,2,'2023-01-12','application_start','Investment Account','mobile'),(9,2,'2023-01-13','application_submit','Investment Account','mobile'),(10,2,'2023-01-15','account_opened','Investment Account','mobile'),(11,3,'2023-01-20','page_view','Chequing Account','web'),(12,3,'2023-01-20','product_view','Chequing Account','web'),(13,3,'2023-01-20','application_start','Chequing Account','web'),(14,4,'2023-02-03','page_view','Chequing Account','web'),(15,4,'2023-02-03','product_view','Chequing Account','web'),(16,5,'2023-02-14','page_view','Savings Account','mobile'),(17,5,'2023-02-14','product_view','Savings Account','mobile'),(18,5,'2023-02-14','application_start','Savings Account','mobile'),(19,5,'2023-02-15','application_submit','Savings Account','mobile'),(20,5,'2023-02-20','account_opened','Savings Account','mobile'),(21,7,'2023-03-10','page_view','Investment Account','web'),(22,7,'2023-03-10','product_view','Investment Account','web'),(23,7,'2023-03-10','application_start','Investment Account','web'),(24,7,'2023-03-11','application_submit','Investment Account','web'),(25,7,'2023-03-15','account_opened','Investment Account','web'),(26,8,'2023-04-05','page_view','Chequing Account','mobile'),(27,8,'2023-04-05','product_view','Chequing Account','mobile'),(28,9,'2023-04-18','page_view','Investment Account','web'),(29,9,'2023-04-18','product_view','Investment Account','web'),(30,9,'2023-04-18','application_start','Investment Account','web'),(31,9,'2023-04-20','application_submit','Investment Account','web'),(32,9,'2023-04-25','account_opened','Investment Account','web');""")
conn.commit()
print("Finance schema ready.")
for t in ["customers","accounts","transactions","product_events"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n} rows")

---
## Monthly activity flags per customer

In [ ]:
print("=== Monthly activity: did each customer transact in each month? ===")
q = """
WITH months AS (
    -- All distinct transaction months in the dataset
    SELECT DISTINCT STRFTIME('%Y-%m', txn_date) AS month
    FROM   transactions
),
customer_months AS (
    -- Cross join: every customer x every month
    SELECT c.customer_id, c.first_name || ' ' || c.last_name AS customer,
           c.signup_date, m.month
    FROM   customers AS c CROSS JOIN months AS m
),
active AS (
    SELECT DISTINCT customer_id, STRFTIME('%Y-%m', txn_date) AS active_month
    FROM   transactions
)
SELECT  cm.customer_id,
        cm.customer,
        cm.month,
        CASE WHEN a.customer_id IS NOT NULL THEN 1 ELSE 0 END  AS was_active
FROM    customer_months AS cm
LEFT JOIN active AS a
       ON cm.customer_id = a.customer_id AND cm.month = a.active_month
WHERE   cm.month >= STRFTIME('%Y-%m', cm.signup_date)
ORDER BY cm.customer_id, cm.month
"""
df = pd.read_sql(q, conn)
# Pivot for readability
pivot = df.pivot(index='customer', columns='month', values='was_active').fillna('')
print(pivot.to_string())


---
## Rolling monthly retention by cohort

In [ ]:
print("=== Rolling retention: % of cohort active in each month ===")
q = """
WITH cohorts AS (
    SELECT customer_id,
           CASE WHEN STRFTIME('%m',signup_date) IN ('01','02','03') THEN 'Q1-2023'
                WHEN STRFTIME('%m',signup_date) IN ('04','05','06') THEN 'Q2-2023'
                ELSE 'Q3-2023' END AS cohort,
           signup_date
    FROM   customers
),
cohort_sizes AS (
    SELECT cohort, COUNT(*) AS cohort_size FROM cohorts GROUP BY cohort
),
active AS (
    SELECT DISTINCT customer_id, STRFTIME('%Y-%m', txn_date) AS active_month
    FROM   transactions
),
cohort_activity AS (
    SELECT c.cohort, cs.cohort_size, a.customer_id, a.active_month,
           CAST((JULIANDAY(a.active_month||'-01') - JULIANDAY(c.signup_date))/30 AS INTEGER) AS m
    FROM   cohorts c
    JOIN   cohort_sizes cs ON c.cohort = cs.cohort
    JOIN   active a ON c.customer_id = a.customer_id
    WHERE  CAST((JULIANDAY(a.active_month||'-01') - JULIANDAY(c.signup_date))/30 AS INTEGER) BETWEEN 0 AND 9
)
SELECT  cohort,
        MAX(cohort_size)  AS size,
        m                 AS months_since_signup,
        COUNT(DISTINCT customer_id)  AS retained,
        ROUND(100.0 * COUNT(DISTINCT customer_id) / MAX(cohort_size), 1) AS retention_pct
FROM    cohort_activity
GROUP BY cohort, m
ORDER BY cohort, m
"""
df = pd.read_sql(q, conn)
print(df.to_string(index=False))


---
## Identifying churned customers

In [ ]:
print("=== Churn identification: no transaction in 90+ days ===")
q = """
WITH last_txn AS (
    SELECT  customer_id, MAX(txn_date)  AS last_txn_date
    FROM    transactions
    GROUP BY customer_id
),
observation_date AS (
    SELECT MAX(txn_date) AS obs_date FROM transactions
)
SELECT  c.customer_id,
        c.first_name || ' ' || c.last_name  AS customer,
        c.segment,
        c.signup_date,
        lt.last_txn_date,
        CAST(JULIANDAY((SELECT obs_date FROM observation_date))
             - JULIANDAY(lt.last_txn_date) AS INTEGER)  AS days_since_last_txn,
        CASE
            WHEN CAST(JULIANDAY((SELECT obs_date FROM observation_date))
                      - JULIANDAY(lt.last_txn_date) AS INTEGER) >= 90
            THEN 'Churned'
            ELSE 'Active'
        END  AS churn_status
FROM    customers AS c
LEFT JOIN last_txn AS lt ON c.customer_id = lt.customer_id
ORDER BY days_since_last_txn DESC NULLS LAST
"""
df = pd.read_sql(q, conn)
print(df.to_string(index=False))
print()
churn = (df['churn_status'] == 'Churned').sum()
total = len(df)
print(f"Churn rate (90-day): {churn}/{total} = {100*churn/total:.1f}%")
conn.close()


---
## Common Pitfalls

**1. Defining "active" too loosely or too strictly**
"Active" means different things in different products. A payroll deposit every month is active for a bank. A login is active for an app. One transaction in a 90-day window may be the right threshold for some products, wrong for others. Document and consistently apply your definition.

**2. Confusing day-N retention with rolling retention**
Day-N (classic) retention asks: was the user active on exactly day N? Rolling retention asks: was the user active at any point from day 0 through day N? Rolling retention is always >= classic retention. Reporting one when stakeholders expect the other causes misunderstanding.

**3. Using calendar months instead of cohort-relative months**
Comparing January activity across cohorts that signed up at different times conflates cohort age with calendar time. Month 1 must mean "one month after each cohort's acquisition date," not "January" for everyone.

**4. Not separating voluntary churn from involuntary churn**
A customer who closed their account is different from a customer who simply stopped transacting. Merging active=0 (closed) with last_txn > 90 days produces a churn rate that mixes two distinct phenomena. Segment churn by reason where possible.

**5. Calling a customer churned based on the observation date**
If the dataset ends October 31 and a customer last transacted October 25, they appear about to churn -- but they may have transacted November 1. Always document the observation date and treat customers near the edge of the window as right-censored rather than definitively churned.

**6. Ignoring the survival bias in mature cohorts**
The Q1 cohort has had more time to churn than the Q3 cohort. Comparing their raw active counts without controlling for cohort age makes newer cohorts look worse. Always express retention as a percentage of cohort size, not as a raw count.


---
*sql_methods_library - Samantha McGarrigle*